# 37 — M4 Gate

Gate computable de Modeling & Evaluation. Comprueba los componentes 30–36, lineage, política de aislamiento de TEST, hashes de la selección congelada y persistencia del model card.

In [ ]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m3_gate = paths.docs / "decisions" / "M3_GATE.md"
assert m3_gate.exists() and "M3 ✅ CLOSED" in m3_gate.read_text(encoding="utf-8"), (
    "M3 no está formalmente cerrado. No iniciar Modeling."
)
GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
MODEL_DIR = paths.root / "artifacts" / "models" / "m4"
MANIFEST_DIR = paths.root / "artifacts" / "manifests"

{
    "gold_schema": GOLD_SPEC.schema_version,
    "modeling_schema": MODEL_SPEC.schema_version,
    "target": MODEL_SPEC.target,
    "model_matrix": MODEL_SPEC.model_matrix,
    "operational_review_fraction": MODEL_SPEC.ranking.operational_fraction,
}

from argos.evaluation.m4_gate import evaluate_m4_gate, persist_m4_gate


In [ ]:
report, decision = evaluate_m4_gate(
    root=paths.root,
    reports_root=paths.reports,
    model_dir=MODEL_DIR,
    manifest_dir=MANIFEST_DIR,
    docs_root=paths.docs,
    schema_version=MODEL_SPEC.schema_version,
)
report


In [ ]:
report_path, decision_path = persist_m4_gate(
    report=report,
    decision=decision,
    reports_root=paths.reports,
    decision_path=paths.docs / "decisions" / "M4_GATE.md",
)
print(f"M4 {'✅ CLOSED' if decision.closed else '❌ OPEN'}")
print(f"Blocking checks: {decision.blocking_passed}/{decision.blocking_total}")
print("Gate report:", report_path)
print("Decision artifact:", decision_path)
assert decision.closed, f"M4 permanece OPEN: {decision.failed_checks}"
